# 04 Sector Clustering
## Nifty Financial Platform

This notebook applies K-Means clustering to group companies based on their financial profiles. We use PCA (Principal Component Analysis) to visualize these clusters in 2D.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from dotenv import load_dotenv

# Settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load environment variables
load_dotenv(dotenv_path='../.env')
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    print("ERROR: DATABASE_URL not found in .env file")
else:
    engine = create_engine(DATABASE_URL)
    print("Database engine created successfully.")

## 1. Prepare Data for Clustering

In [ ]:
def fetch_clustering_data(engine):
    query = """
    SELECT 
        dc.symbol, dc.company_name, ds.sector_name, 
        pl.net_profit_margin_pct, 
        bs.debt_to_equity,
        an.stock_pe, an.roce_pct, an.dividend_yield
    FROM fact_profit_loss pl
    JOIN dim_company dc ON pl.company_id = dc.company_id
    JOIN dim_sector ds ON dc.sector_id = ds.sector_id
    JOIN fact_balance_sheet bs ON pl.company_id = bs.company_id AND pl.year_id = bs.year_id
    JOIN fact_analysis an ON pl.company_id = an.company_id AND pl.year_id = an.year_id
    WHERE pl.year_id = (SELECT MAX(year_id) FROM fact_profit_loss)
    """
    return pd.read_sql(query, engine)

df = fetch_clustering_data(engine).dropna()
features = ['net_profit_margin_pct', 'debt_to_equity', 'stock_pe', 'roce_pct', 'dividend_yield']
df.head()

## 2. Optimal Number of Clusters (Elbow Method)

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(df[features])

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

## 3. Perform K-Means Clustering

In [ ]:
k = 4 # Selected based on elbow plot
kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42)
df['cluster'] = kmeans.fit_predict(X)

print(f"Clustering complete with {k} clusters.")
df['cluster'].value_counts()

## 4. PCA for Visualization
Reducing 5 dimensions to 2 for visualization.

In [ ]:
pca = PCA(n_components=2)
pca_results = pca.fit_transform(X)
df['pca1'] = pca_results[:, 0]
df['pca2'] = pca_results[:, 1]

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df, x='pca1', y='pca2', hue='cluster', palette='viridis', s=100, alpha=0.7)
plt.title('Company Clusters (PCA Visualization)')
plt.xlabel(f'PCA 1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PCA 2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.show()

## 5. Cluster Characterization
What defines each cluster?

In [ ]:
cluster_profile = df.groupby('cluster')[features].mean()
sns.heatmap(cluster_profile, annot=True, cmap='YlGnBu')
plt.title('Average Metrics per Cluster')
plt.show()

## 6. Sample Companies per Cluster

In [ ]:
for i in range(k):
    print(f"\n--- Cluster {i} Companies ---")
    print(df[df['cluster'] == i]['company_name'].head(5).tolist())